In [1]:
# Modules
from utils import wiki_scraper, convert_to_decimal_degrees, pop_density, world_pop
import pandas as pd
import geopandas as gpd
from sklearn.neighbors import BallTree
from scipy.spatial import cKDTree
import numpy as np

In [2]:

def validation_dataset(landsat='data/validate/landsat_features_validation.csv', terraclimate='data/validate/terraclimate_features_validation.csv', col='Sample Date'):
    '''
    input: 2 validation datasets (CSV files) from EY's Water Quality Prediction Benchmark Notebook
    return: merged pandas dataframe with added bin column (monthly) joined on GPS coordinates and date
    '''
    # read CSVs and Handling Missing Values
    landsat = pd.read_csv(landsat)
    # landsat.fillna(landsat.median(numeric_only=True))
    terraclimate = pd.read_csv(terraclimate)

    # merge features and targets on coordinates and date; convert to datetime
    landsat_terracl = pd.merge(landsat, terraclimate)
    landsat_terracl[col] = pd.to_datetime(landsat_terracl[col], dayfirst=True, errors='coerce')

    # bin according to month
    bins = pd.date_range(start=landsat_terracl[col].min(), end=landsat_terracl[col].max(), freq='ME')
    landsat_terracl['month'] = pd.cut(landsat_terracl[col], bins=len(bins), labels=bins)

    # Handling Missing Values
    val_data = landsat_terracl.fillna(landsat_terracl.median(numeric_only=True))
    print('null values:','\n',val_data.isna().sum())
    val_data.columns = landsat_terracl.columns.str.strip().str.lower()

    print('We will validate Water Quality over the course of',len(val_data[['month']].groupby('month', observed=True).count()),'months.')

    return val_data

In [3]:
val_data = validation_dataset()
val_data.head()

null values: 
 Latitude       0
Longitude      0
Sample Date    0
nir            0
green          0
swir16         0
swir22         0
NDMI           0
MNDWI          0
pet            0
month          0
dtype: int64
We will validate Water Quality over the course of 55 months.


,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,mndwi,pet,month
0,-32.043333,27.822778,2014-09-01,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727,161.90001,2014-08-31
1,-33.329167,26.077500,2015-09-16,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,177.60000,2015-08-31
2,-32.991639,27.640028,2015-05-07,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979,158.40001,2015-04-30
3,-34.096389,24.439167,2012-02-07,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,130.00000,2012-01-31
4,-32.000556,28.581667,2014-10-01,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052,152.50000,2014-09-30


In [4]:
# Scrape a list of rivers from South Africa @ https://en.wikipedia.org/wiki/List_of_rivers_of_South_Africa
# Custom headers to mimic a real browser
# def wiki_scraper(url="https://en.wikipedia.org/wiki/List_of_rivers_of_South_Africa", headers={"User-Agent": "Chrome/107.0.0.0 Safari/537.36"}, table=1, subset='Mouth / junction coordinates'):

rivers = wiki_scraper.wiki_scraper(url='https://en.wikipedia.org/wiki/List_of_rivers_of_South_Africa')

# Convert Degrees, Minutes and Seconds to Decimal Degrees
# Formula: Decimal Degrees = Degrees + (Minutes ÷ 60) + (Seconds ÷ 3,600)
# For example, to convert 30° 15′ 50″: Decimal Degrees = 30 + (15 ÷ 60) + (50 ÷ 3,600) = 30.2639°.
# https://stackoverflow.com/questions/33997361/how-to-convert-degree-minute-second-to-degree-decimal

# extensive 'print' debugging to figure out '\ufeff' was at the end of each string in the pd.Series
rivers['mouth/junctioncoordinates'] = rivers['mouth/junctioncoordinates'].apply(lambda x: x.replace('\ufeff', ""))
rivers['latitude'] = convert_to_decimal_degrees.convert_to_decimal_degrees(rivers['mouth/junctioncoordinates'])['latitude']
rivers['longitude'] = convert_to_decimal_degrees.convert_to_decimal_degrees(rivers['mouth/junctioncoordinates'])['longitude']
rivers= rivers[rivers['latitude'].apply(lambda x: isinstance(x, float))]

# round to 5 decimals
rivers['latitude'] = round(rivers['latitude'].astype(float), 5)
rivers['longitude'] = round(rivers['longitude'].astype(float), 5)

# all river names lower case
rivers['river'] = rivers['river'].str.lower()


rivers.info()

<class 'pandas.DataFrame'>
Index: 50 entries, 0 to 274
Data columns (total 10 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   river                           50 non-null     str    
 1   drainagebasin[a]                29 non-null     str    
 2   provinceandlocation             50 non-null     str    
 3   sourcelocation(town/mountains)  36 non-null     str    
 4   tributaryof(river)              25 non-null     str    
 5   daminriver                      19 non-null     str    
 6   mouth/junctionatlocation(town)  43 non-null     str    
 7   mouth/junctioncoordinates       50 non-null     str    
 8   latitude                        50 non-null     float64
 9   longitude                       50 non-null     float64
dtypes: float64(2), str(8)
memory usage: 4.3 KB


In [5]:
river_mouths = pd.read_excel('data/south_africa_river_mouths.xlsx')
river_mouths['latitude'] = round(river_mouths['latitude'], 5)
river_mouths['longitude'] = round(river_mouths['longitude'], 5)

# all river_names lowercase
river_mouths['river'] = river_mouths['river_name'].str.lower()
river_mouths = river_mouths[['river_name','province','mouth_location','latitude','longitude','river']] ######
river_mouths.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   river_name      21 non-null     str    
 1   province        21 non-null     str    
 2   mouth_location  21 non-null     str    
 3   latitude        21 non-null     float64
 4   longitude       21 non-null     float64
 5   river           21 non-null     str    
dtypes: float64(2), str(4)
memory usage: 1.1 KB


In [6]:
# Let's create a function for loading in the next datasets.
# We will create 2 GeoDataFrames and perform a spatial join with our primary dataset.

def shapeF(df, col='ADMIN', path_to_file=str, layer=None) -> pd.DataFrame:
    '''
    input:
    ------
    - df: pandas dataframe
    - col: str (column name)

    returns
    -------
    joined and cleaned pandas dataframe
    '''
    
    if col == 'ADMIN':
        new_col_name = 'country'
    else:
        new_col_name = 'province'

    print(new_col_name)

    # 1) Create GeoDataFrame from lat/lon
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326" # "Coordinate Reference System"
    )

    # 2) Load Natural Earth country boundaries
    count = gpd.read_file(path_to_file, layer=layer) # other dependent files (.shx, .dhp, etc.) are located in this path*
    #.to_crs("EPSG:4326")
    # WARNINGS SUPPRESSED

    # 3) Spatial join to add country column
    gdf_join = gpd.sjoin(
        gdf,
        count[[col, "geometry"]],
        how="left",
        predicate="within"
    ).to_crs("EPSG:4326")

    # 4) Rename + cleanup country columns
    gdf_join = (
        gdf_join
        .rename(columns={col: new_col_name})
        .drop(columns=["geometry", "index_right"], errors="ignore")
    )

    # 7) Move `country` to be the FIRST column
    cols = gdf_join.columns.tolist()
    cols.insert(0, cols.pop(cols.index(new_col_name)))
    gdf_join = gdf_join[cols]

    # 8) Quick check
    return gdf_join

In [7]:
# define variables for our function
cols_and_paths = {'ADMIN': 'data/new_earth/ne_50m_admin_0_countries.shp', 'name': 'data/za_shp'}

gdf_with_country = shapeF(val_data, col='ADMIN', path_to_file=cols_and_paths['ADMIN'])
final_df = shapeF(gdf_with_country, col='name', path_to_file=cols_and_paths['name'], layer='za')
final_df.head()

country
province


,province,country,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,mndwi,pet,month
0,Eastern Cape,South Africa,-32.043333,27.822778,2014-09-01,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727,161.90001,2014-08-31
1,Eastern Cape,South Africa,-33.329167,26.077500,2015-09-16,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,177.60000,2015-08-31
2,Eastern Cape,South Africa,-32.991639,27.640028,2015-05-07,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979,158.40001,2015-04-30
3,Eastern Cape,South Africa,-34.096389,24.439167,2012-02-07,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,130.00000,2012-01-31
4,Eastern Cape,South Africa,-32.000556,28.581667,2014-10-01,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052,152.50000,2014-09-30


In [8]:
# Follow the yellow-brick-road
path = 'data/Population_density'
filenames = pop_density.pop_density()
filenames
# Source - https://stackoverflow.com/q/20906474
# Posted by jonas, modified by community. See post 'Timeline' for change history
# Retrieved 2026-01-30, License - CC BY-SA 4.0

['data/Population_density\\zaf_pd_2011_1km_UNadj_ASCII_XYZ.zip',
 'data/Population_density\\zaf_pd_2012_1km_UNadj_ASCII_XYZ.zip',
 'data/Population_density\\zaf_pd_2013_1km_UNadj_ASCII_XYZ.zip',
 'data/Population_density\\zaf_pd_2014_1km_UNadj_ASCII_XYZ.zip',
 'data/Population_density\\zaf_pd_2015_1km_UNadj_ASCII_XYZ.zip']

In [9]:
all_dfs = [world_pop.world_pop(filename) for filename in filenames]

# Combine all countries & years into one DataFrame
population_density_all = pd.concat(all_dfs, ignore_index=True)

# Sort for consistency
population_density_all = population_density_all.sort_values(
    by=["country", "year", "latitude", "longitude", "population_density"]
).reset_index(drop=True)

# Quick check before joining; you'll notice the countries are all "Unknown"
population_density_all.head()

Population density from 2011 ['zaf', 'pd', '2011', '1km', 'UNadj', 'ASCII', 'XYZ.zip']
Population density from 2012 ['zaf', 'pd', '2012', '1km', 'UNadj', 'ASCII', 'XYZ.zip']
Population density from 2013 ['zaf', 'pd', '2013', '1km', 'UNadj', 'ASCII', 'XYZ.zip']
Population density from 2014 ['zaf', 'pd', '2014', '1km', 'UNadj', 'ASCII', 'XYZ.zip']
Population density from 2015 ['zaf', 'pd', '2015', '1km', 'UNadj', 'ASCII', 'XYZ.zip']


,country,year,longitude,latitude,population_density
0,South Africa,2011,37.794583,-46.979583,0.0
1,South Africa,2011,37.802917,-46.979583,0.0
2,South Africa,2011,37.811250,-46.979583,0.0
3,South Africa,2011,37.819583,-46.979583,0.0
4,South Africa,2011,37.827917,-46.979583,0.0


In [10]:
# data_val = val_data.copy()
data_val = final_df.copy()
pd_all = population_density_all.copy()

data_val.columns = data_val.columns.str.strip().str.lower()
pd_all.columns = pd_all.columns.str.strip().str.lower()

# --- 2) Ensure numeric types ---
for col in ["longitude", "latitude"]:
    data_val[col] = pd.to_numeric(data_val[col], errors="coerce")
    pd_all[col] = pd.to_numeric(pd_all[col], errors="coerce")

pd_all["population_density"] = pd.to_numeric(pd_all["population_density"], errors="coerce")
pd_all["year"] = pd.to_numeric(pd_all["year"], errors="coerce")

# --- 3) Extract sample year ---
data_val["sample_year"] = pd.to_datetime(data_val["sample date"], errors="coerce").dt.year

# --- 4) Initialize output columns ---
data_val["pd_year"] = np.nan
data_val["pop_density_nn"] = np.nan
data_val["distance_km_to_pd_cell"] = np.nan

data_val.head()

,province,country,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,mndwi,pet,month,sample_year,pd_year,pop_density_nn,distance_km_to_pd_cell
0,Eastern Cape,South Africa,-32.043333,27.822778,2014-09-01,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727,161.90001,2014-08-31,2014,NaN,NaN,NaN
1,Eastern Cape,South Africa,-33.329167,26.077500,2015-09-16,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,177.60000,2015-08-31,2015,NaN,NaN,NaN
2,Eastern Cape,South Africa,-32.991639,27.640028,2015-05-07,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979,158.40001,2015-04-30,2015,NaN,NaN,NaN
3,Eastern Cape,South Africa,-34.096389,24.439167,2012-02-07,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,130.00000,2012-01-31,2012,NaN,NaN,NaN
4,Eastern Cape,South Africa,-32.000556,28.581667,2014-10-01,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052,152.50000,2014-09-30,2014,NaN,NaN,NaN


In [11]:
# --- 5) Helper: approx distance in km ---
def approx_deg_to_km(dlon, dlat, lat):
    '''
    input
    -----
    dlon: distance between 2 longitudinal lines
    dlat: distance between 2 latitudinal lines
    lat: distance between 
    return
    ------
    float: distance between 2 points in km
    '''
    km_lat = dlat * 111.32
    km_lon = dlon * 111.11 * np.cos(np.deg2rad(lat))
    return np.sqrt(km_lat**2 + km_lon**2)

# --- 6) Filter PD to clean rows only ---
pd_all_clean = pd_all.dropna(subset=["year", "longitude", "latitude", "population_density"]).copy()
pd_all_clean["year"] = pd_all_clean["year"].astype(int)
print("There are ",len(pd_all_clean),"population densities")
print(len(pd_all_clean[pd_all_clean['country'].isna()==True]),"of them are null")


# --- 7) Build KDTree per year once ---
trees = {}
pd_by_year = {}

# Group population density grid cells by year and reset index so positional indexing line up with KDTree output
# Use coordinates for nearest-neighbor matching and build KDTree for fast nearest point search
for yr, group in pd_all_clean.groupby("year"):
    group = group.reset_index(drop=True)
    coords = group[["longitude", "latitude"]].to_numpy()
    trees[yr] = cKDTree(coords)
    # Save the year-specific dataframe in pd_by_year dictionary
    pd_by_year[yr] = group

# --- 8) Match WQ points year-by-year ---
# Keep only sampling rows with valid year + coordinates
valid_wq = data_val.dropna(subset=["sample_year", "longitude", "latitude"]).copy()
valid_wq["sample_year"] = valid_wq["sample_year"].astype(int)

# Loop through each year that appears in the sampling data
# If population data doesn't exist for this year, skip
for yr in sorted(valid_wq["sample_year"].unique()):
    if yr not in trees:
        print(yr,'not in trees')
        continue

    # Indices for sampling points this year
    # Coordinates for sampling points this year
    idx_rows = valid_wq.index[valid_wq["sample_year"] == yr]
    query_pts = valid_wq.loc[idx_rows, ["longitude", "latitude"]].to_numpy()

    # KDTree nearest-neighbor query:
    # dist -> distance in degrees (not km)
    # idx -> index of nearest population grid cell row (within pd_by_year[yr])
    # Retrieve the matched population density rows
    dist, idx = trees[yr].query(query_pts, k=1)
    matched = pd_by_year[yr].iloc[idx].reset_index(drop=True)

    # Store matched year + population density back into the original wq dataframe
    data_val.loc[idx_rows, "pd_year"] = yr
    data_val.loc[idx_rows, "pop_density_nn"] = matched["population_density"].to_numpy()

    # Compute km distance between sampling point and matched grid cell centroid
    dlon = valid_wq.loc[idx_rows, "longitude"].to_numpy() - matched["longitude"].to_numpy()
    dlat = valid_wq.loc[idx_rows, "latitude"].to_numpy() - matched["latitude"].to_numpy()
    lat  = valid_wq.loc[idx_rows, "latitude"].to_numpy()

    data_val.loc[idx_rows, "distance_km_to_pd_cell"] = approx_deg_to_km(dlon, dlat, lat)

print('Population Density has been added for each sample\nIt is the estimated population per square km\n',round(data_val['pop_density_nn'].describe(), None).astype(int))
data_val.head()



There are  8157545 population densities
0 of them are null
Population Density has been added for each sample
It is the estimated population per square km
 count    200
mean      44
std       92
min        0
25%        2
50%       10
75%       45
max      559
Name: pop_density_nn, dtype: int64


,province,country,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,mndwi,pet,month,sample_year,pd_year,pop_density_nn,distance_km_to_pd_cell
0,Eastern Cape,South Africa,-32.043333,27.822778,2014-09-01,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727,161.90001,2014-08-31,2014,2014.0,105.844948,0.442666
1,Eastern Cape,South Africa,-33.329167,26.077500,2015-09-16,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,177.60000,2015-08-31,2015,2015.0,2.098670,0.060331
2,Eastern Cape,South Africa,-32.991639,27.640028,2015-05-07,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979,158.40001,2015-04-30,2015,2015.0,39.784103,0.543766
3,Eastern Cape,South Africa,-34.096389,24.439167,2012-02-07,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,130.00000,2012-01-31,2012,2012.0,1.452179,0.268849
4,Eastern Cape,South Africa,-32.000556,28.581667,2014-10-01,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052,152.50000,2014-09-30,2014,2014.0,53.753334,0.570846


In [13]:
# --- 9) Build FINAL joined dataset: all original columns + new features ---
# original_cols = val_data.copy()
original_cols = final_df.copy()
original_cols.columns = original_cols.columns.str.strip().str.lower()
original_cols = original_cols.columns.tolist()

new_cols = ["sample_year", "pop_density_nn", "distance_km_to_pd_cell"]

# Keep original order + add new cols at the end (only if they exist)
final_cols = original_cols + [c for c in new_cols if c in data_val.columns]
wq_joined = data_val[final_cols].copy()

# --- 10) Take a peak ---
wq_joined.head()

,province,country,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,mndwi,pet,month,sample_year,pop_density_nn,distance_km_to_pd_cell
0,Eastern Cape,South Africa,-32.043333,27.822778,2014-09-01,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727,161.90001,2014-08-31,2014,105.844948,0.442666
1,Eastern Cape,South Africa,-33.329167,26.077500,2015-09-16,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,177.60000,2015-08-31,2015,2.098670,0.060331
2,Eastern Cape,South Africa,-32.991639,27.640028,2015-05-07,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979,158.40001,2015-04-30,2015,39.784103,0.543766
3,Eastern Cape,South Africa,-34.096389,24.439167,2012-02-07,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,130.00000,2012-01-31,2012,1.452179,0.268849
4,Eastern Cape,South Africa,-32.000556,28.581667,2014-10-01,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052,152.50000,2014-09-30,2014,53.753334,0.570846


In [14]:
rivers = pd.read_pickle('data/rivers.pkl')
river_mouths = pd.read_pickle('data/river_mouths.pkl')

# distinguish river mouths from junctions
river_junctions = rivers[~rivers['river'].isin(river_mouths['river'])]
river_junctions.head()

,river,drainagebasin[a],provinceandlocation,sourcelocation(town/mountains),tributaryof(river),daminriver,mouth/junctionatlocation(town),mouth/junctioncoordinates,latitude,longitude
0,amandawe river,NaN,"Amandawe, KwaZulu-Natal",north west of Amandawe,north of Scottburgh,NaN,Umpambanyoni River,30°15′30″S 30°45′0″E / 30.25833°S 30.75000°E,-30.25833,30.75000
1,amahlongwa river,NaN,"Amahlongwa, KwaZulu-Natal",south west of Amahlongwa,west of uMkomaas,NaN,Indian Ocean,30°15′30″S 30°44′0″E / 30.25833°S 30.73333°E,-30.25833,30.73333
3,apies river,A2,"Gauteng, Tshwane, Pretoria","Bronberg, southeast of Pretoria","Moretele River, then Crocodile River and Limpo...",Bon Accord Dam,Makapanstad,25°14′24″S 28°08′36″E / 25.24000°S 28.14333°E,-25.24000,28.14333
4,as river (or axel river),C8,Free State,Southeast of Bethlehem,"Liebenbergsvlei River, then Wilge River",Sol Plaatjie Dam,NaN,28°13′27″S 28°21′58″E / 28.22417°S 28.36611°E,-28.22417,28.36611
5,assegaai river,W5,Mpumalanga,North of Wakkerstroom,Mkondo River,Heyshope Dam,Swaziland border,27°04′46″S 31°02′19″E / 27.07944°S 31.03861°E,-27.07944,31.03861


In [15]:
def rivers_end(df, THRESH_KM: int, new_col=str) -> pd.DataFrame:
    '''
    input
    -----
    - df: pandas dataframe
    - THRESH_KM: int
    - new_col: str (name of the column that'll be added

    return
    ------
    pandas dataframe with binary column added
    '''
    THRESH_KM = THRESH_KM
    EARTH_RADIUS_KM = 6371

    # river mouth coordinates
    mouth_coords = np.radians(
        df[["latitude", "longitude"]].astype(float).dropna().to_numpy()
    )

    # establish tree, and use the "haversine" distance metric for the geospatial nearest neighbor search
    tree = BallTree(mouth_coords, metric="haversine")

    # sampling point coordinates
    sample_coords = np.radians(
        wq_joined[["latitude", "longitude"]].to_numpy()
    )

    # query the tree for its nearest neighbors
    dist_rad, _ = tree.query(sample_coords, k=1)

    # dist_rad is in the wrong shape
    # print("dist_rad",dist_rad.shape)
    # print("index",_.shape)

    # binary classification
    wq_joined[new_col] = (
        dist_rad[:, 0] * EARTH_RADIUS_KM <= THRESH_KM
    ).astype(int)

    return wq_joined

river_mouthORjunction_df = rivers_end(rivers, THRESH_KM=10, new_col='river_mouthORjunction')
river_mouth_df = rivers_end(river_mouths, THRESH_KM=10, new_col='river_mouth')
river_junction_df = rivers_end(river_junctions, THRESH_KM=10, new_col='river_junction')

In [16]:
# eloquently sort values while observing if there are any missing values
wq_joined = river_junction_df.copy()
# wq_joined = wq_joined.dropna(subset='province')
print('shape:',wq_joined.shape)
wq_joined.to_csv('data/validate.csv', index=False)
wq_joined.isna().sum().sort_values(ascending=False)


shape: (200, 19)


province                  0
country                   0
latitude                  0
longitude                 0
sample date               0
nir                       0
green                     0
swir16                    0
swir22                    0
ndmi                      0
mndwi                     0
pet                       0
month                     0
sample_year               0
pop_density_nn            0
distance_km_to_pd_cell    0
river_mouthORjunction     0
river_mouth               0
river_junction            0
dtype: int64

In [17]:
wq_joined.tail()

,province,country,latitude,longitude,sample date,nir,green,swir16,swir22,ndmi,mndwi,pet,month,sample_year,pop_density_nn,distance_km_to_pd_cell,river_mouthORjunction,river_mouth,river_junction
195,Eastern Cape,South Africa,-33.771111,25.386667,2012-12-06,17562.0,9492.0,13559.5,10235.0,0.128609,-0.176453,171.400010,2012-11-30,2012,472.031372,0.041501,0,0,0
196,Eastern Cape,South Africa,-33.185361,27.390750,2014-09-04,15883.0,9083.5,12135.5,9484.0,0.133751,-0.143833,159.400010,2014-08-31,2014,30.141207,0.456029,0,0,0
197,Eastern Cape,South Africa,-32.043333,27.822778,2015-09-28,13619.5,10046.5,13105.0,10969.0,0.019252,-0.132108,168.600000,2015-09-30,2015,97.786865,0.442666,0,0,0
198,Eastern Cape,South Africa,-33.001667,25.161389,2015-01-08,13955.5,10670.0,17303.5,14835.5,-0.107105,-0.237135,81.200005,2014-12-31,2015,2.641395,0.324866,0,0,0
199,Eastern Cape,South Africa,-33.237780,26.994720,2013-03-27,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,152.800000,2013-03-31,2013,2.146590,0.019794,0,0,0
